# CIP alerting system

This notebook compares actual CIP cycles from the log files with the object-specific thresholds in `CIP Pre Rinse Timer details.xlsx`.

A cycle is defined as a continuous block of records for one object. A gap of more than 30 minutes starts the next cycle. The log's `Time In Seconds` is cumulative within a cycle, so each activity duration is calculated as its ending counter minus its starting counter.


In [9]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

DATA_DIR = Path('datasets')
LOG_FILES = sorted(DATA_DIR.glob('CIP_Log*.xlsx'))
REFERENCE_FILE = DATA_DIR / 'CIP Pre Rinse Timer details.xlsx'

# These are deliberately visible and editable.
CYCLE_GAP_MINUTES = 30
DURATION_TOLERANCE = 0.10  # Alert when actual duration exceeds threshold + 10%
FLOW_TOLERANCE = 0.10      # 10% below the reference flow target

print(f'{len(LOG_FILES)} log files found')
print(f'Reference workbook: {REFERENCE_FILE.name}')

if not LOG_FILES:
    raise FileNotFoundError(f'No CIP log files found in {DATA_DIR.resolve()}')
if not REFERENCE_FILE.exists():
    raise FileNotFoundError(f'Reference workbook not found: {REFERENCE_FILE.resolve()}')


4 log files found
Reference workbook: CIP Pre Rinse Timer details.xlsx


In [10]:
REQUIRED = [
    'Date Time', 'Object', 'Recipe', 'Activity', 'Step',
    'Time In Seconds', 'Temp SP', 'Return Temp', 'Supply Flow',
    'Return Flow', 'Min Return Flow SP', 'Return Conducivity',
    'Caustic SP', 'Acid SP', 'Final Rinse Cond. SP', 'Flip'
]

def line_from_report(path):
    raw = pd.read_excel(path, header=None, nrows=10)
    text = ' '.join(raw.astype(str).fillna('').stack())
    match = re.search(r'Line\s*:\s*([^ ]+)', text, flags=re.I)
    return match.group(1) if match else path.stem

def read_log(path):
    df = pd.read_excel(path, sheet_name=0, header=13)
    df = df[REQUIRED].copy()
    df['Date Time'] = pd.to_datetime(df['Date Time'], errors='coerce')
    df = df[df['Date Time'].notna()].copy()
    for col in REQUIRED[5:]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['Line'] = line_from_report(path)
    df['Source File'] = path.name
    return df

logs = pd.concat([read_log(path) for path in LOG_FILES], ignore_index=True)
logs = logs.sort_values(['Line', 'Object', 'Date Time']).reset_index(drop=True)
print(f'{len(logs):,} timestamped log rows loaded')
logs[['Line', 'Object', 'Activity', 'Step']].head()


38,943 timestamped log rows loaded


,Line,Object,Activity,Step
0,M01W01,C01C01-RMT Tank-1,Pre Rinse,Fill To Tank
1,M01W01,C01C01-RMT Tank-1,Pre Rinse,Fill To Tank
2,M01W01,C01C01-RMT Tank-1,Pre Rinse,Fill Circuit
3,M01W01,C01C01-RMT Tank-1,Pre Rinse,Circulation
4,M01W01,C01C01-RMT Tank-1,Pre Rinse,Circulation


In [11]:
summary = pd.read_excel(REFERENCE_FILE, sheet_name='Summary')
process = pd.read_excel(REFERENCE_FILE, sheet_name='Process CIP Water Consumption')

def object_key(value):
    # Uses the equipment code shared by the reference and log names, e.g. C02C01.
    match = re.search(r'C\d{2}[A-Z]\d{2}', str(value).upper())
    return match.group(0) if match else str(value).strip().upper()

summary['Object Key'] = summary['Object'].map(object_key)
process['Object Key'] = process['Object'].map(object_key)
logs['Object Key'] = logs['Object'].map(object_key)

final_rinse_seconds = [
    'Final Rinse Fill to Tank (sec)',
    'Final Rinse Fill Circuit (Sec)',
    'Final Rinse Front Mix Zone (Sec)',
    'Final Rinse Circulation (Sec)',
      ]

thresholds = summary.merge(
    process[['Object', 'Suppy Flow (L/Hr)', 'Return Flow (L/Hr)', *final_rinse_seconds]],
    on='Object',
    how='left',
    validate='one_to_one'
)
thresholds['Final Rinse Total Time (min)'] = (
    thresholds[final_rinse_seconds].sum(axis=1, min_count=len(final_rinse_seconds)) / 60
)
thresholds = thresholds.rename(columns={
    'New Proposed Pre Rinse total time(Min)': 'Pre Rinse Threshold (min)',
    'Caustic- Total Time': 'Caustic Threshold (min)',
    'Acid Total Time': 'Acid Threshold (min)',
    'Suppy Flow (L/Hr)': 'Supply Flow Threshold (L/hr)',
    'Return Flow (L/Hr)': 'Return Flow Threshold (L/hr)'
})

threshold_view = thresholds[[
    'Object', 'Object Key', 'Pre Rinse Threshold (min)',
    'Caustic Threshold (min)', 'Acid Threshold (min)',
    'Final Rinse Total Time (min)', 'Supply Flow Threshold (L/hr)',
    'Return Flow Threshold (L/hr)'
]].copy()
threshold_view.head()
reference_objects = thresholds[['Object', 'Object Key']].copy()

def reference_object_for_log(log_object):
    key = object_key(log_object)
    candidates = reference_objects[reference_objects['Object Key'].eq(key)]
    if candidates.empty:
        return np.nan
    if len(candidates) == 1:
        return candidates.iloc[0]['Object']
    log_text = str(log_object).upper()
    scores = candidates['Object'].map(
        lambda value: sum(token in log_text for token in re.findall(r'[A-Z0-9]+', str(value).upper()))
    )
    return candidates.loc[scores.idxmax(), 'Object']

logs['Reference Object'] = logs['Object'].map(reference_object_for_log)
print(f"{logs['Reference Object'].notna().sum():,} log rows matched to reference objects")


20,665 log rows matched to reference objects


### Final-rinse calculation

The final-rinse total is calculated as:

`Fill to Tank + Fill Circuit + Front Mix Zone + Circulation`

All four reference fields are in seconds and represent sequential final-rinse steps, so adding them and dividing by 60 is the correct first calculation. If the PLC runs any of these steps in parallel, that assumption should be changed after confirming it with the process owner.


In [12]:
logs = logs.sort_values(['Line', 'Object', 'Date Time']).copy()
new_cycle = (logs.groupby(['Line', 'Object'])['Date Time']
             .diff().gt(pd.Timedelta(minutes=CYCLE_GAP_MINUTES)))
logs['Cycle Number'] = (new_cycle.fillna(True)
                       .groupby([logs['Line'], logs['Object']])
                       .cumsum().astype(int))

cycle_keys = ['Line', 'Object', 'Reference Object', 'Object Key', 'Cycle Number']
cycle_info = (logs.groupby(cycle_keys, as_index=False, dropna=False)
              .agg({'Date Time': ['min', 'max', 'size'],
                    'Recipe': lambda x: ', '.join(sorted(set(x.dropna().astype(str))))}))
cycle_info.columns = cycle_keys + ['Cycle Start', 'Cycle End', 'Record Count', 'Recipes']

activity_times = (logs.groupby(cycle_keys + ['Activity'], as_index=False, dropna=False)
                  .agg({'Time In Seconds': ['min', 'max']}))
activity_times.columns = cycle_keys + ['Activity', 'Start Seconds', 'End Seconds']
activity_times['Activity Time (min)'] = (
    activity_times['End Seconds'] - activity_times['Start Seconds']
) / 60
activity_times.loc[activity_times['Activity Time (min)'] < 0, 'Activity Time (min)'] = np.nan

activity_times = (activity_times
                  .pivot(index=cycle_keys, columns='Activity', values='Activity Time (min)')
                  .reset_index()
                  .rename_axis(None, axis=1))

flow_times = (logs.groupby(cycle_keys, as_index=False, dropna=False)
              .agg({'Supply Flow': 'median', 'Return Flow': ['median', 'min']}))
flow_times.columns = cycle_keys + ['Supply Flow Actual (L/hr)', 'Return Flow Actual (L/hr)', 'Minimum Return Flow Seen (L/hr)']

cycle_results = (cycle_info.merge(activity_times, on=cycle_keys, how='left')
                .merge(flow_times, on=cycle_keys, how='left'))
cycle_results.head()


,Line,Object,Reference Object,Object Key,Cycle Number,Cycle Start,Cycle End,Record Count,Recipes,Acid,Caustic,Final Drain,Final Rinse,Mid Rinse,Pre Rinse,Supply Flow Actual (L/hr),Return Flow Actual (L/hr),Minimum Return Flow Seen (L/hr)
0,M01W01,C01C01-RMT Tank-1,NaN,C01C01,0,2026-07-02 19:38:06.703,2026-07-02 20:14:06.703,37,Caustic,NaN,23.000000,0.0,5.000000,NaN,5.000000,12784.38,20220.0,-1045.0
1,M01W01,C01C01-RMT Tank-1,NaN,C01C01,1,2026-07-04 00:43:06.703,2026-07-04 01:36:06.653,54,Caustic,NaN,38.016667,0.0,6.983333,NaN,5.000000,11404.69,267.5,-50.0
2,M01W01,C01C01-RMT Tank-1,NaN,C01C01,2,2026-07-04 19:54:06.703,2026-07-04 21:14:06.653,81,Caustic Acid,12.0,37.000000,NaN,10.000000,9.0,7.983333,12403.13,3115.0,-1045.0
3,M01W01,C01C01-RMT Tank-1,NaN,C01C01,3,2026-07-06 07:06:06.703,2026-07-06 08:14:06.703,69,Caustic,NaN,51.983333,NaN,10.016667,NaN,4.000000,862.50,-45.0,-1045.0
4,M01W01,C01C01-RMT Tank-1,NaN,C01C01,4,2026-07-07 06:19:06.703,2026-07-07 07:19:06.703,61,Caustic,NaN,46.000000,NaN,7.000000,NaN,5.000000,281.25,-45.0,-560.0


In [13]:
actual_to_threshold = {
    'Pre Rinse': ('Pre Rinse', 'Pre Rinse Threshold (min)'),
    'Caustic': ('Caustic', 'Caustic Threshold (min)'),
    'Acid': ('Acid', 'Acid Threshold (min)'),
    'Final Rinse': ('Final Rinse', 'Final Rinse Total Time (min)')
}

threshold_lookup = thresholds.set_index('Object')

for phase, (actual_col, threshold_col) in actual_to_threshold.items():
    cycle_results[f'{phase} Threshold (min)'] = cycle_results['Reference Object'].map(
        threshold_lookup[threshold_col])
    cycle_results[f'{phase} Breach'] = (
        cycle_results[actual_col].notna()
        & cycle_results[f'{phase} Threshold (min)'].notna()
        & (cycle_results[actual_col] > cycle_results[f'{phase} Threshold (min)'] * (1 + DURATION_TOLERANCE))
    )

cycle_results['Supply Flow Threshold (L/hr)'] = cycle_results['Reference Object'].map(
    threshold_lookup['Supply Flow Threshold (L/hr)'])
cycle_results['Return Flow Threshold (L/hr)'] = cycle_results['Reference Object'].map(
    threshold_lookup['Return Flow Threshold (L/hr)'])
cycle_results['Supply Flow Breach'] = (
    cycle_results['Supply Flow Actual (L/hr)'].notna()
    & cycle_results['Supply Flow Threshold (L/hr)'].notna()
    & (cycle_results['Supply Flow Actual (L/hr)'] < cycle_results['Supply Flow Threshold (L/hr)'] * (1 - FLOW_TOLERANCE))
)
cycle_results['Return Flow Breach'] = (
    cycle_results['Return Flow Actual (L/hr)'].notna()
    & cycle_results['Return Flow Threshold (L/hr)'].notna()
    & (cycle_results['Return Flow Actual (L/hr)'] < cycle_results['Return Flow Threshold (L/hr)'] * (1 - FLOW_TOLERANCE))
)

breach_cols = [c for c in cycle_results.columns if c.endswith('Breach')]
cycle_results['Alert Count'] = cycle_results[breach_cols].sum(axis=1)
cycle_results['Status'] = np.select(
    [cycle_results['Reference Object'].isna(), cycle_results['Alert Count'] > 0],
    ['NO REFERENCE', 'ALERT'],
    default='OK'
)

cycle_summary_columns = cycle_keys + ['Cycle Start', 'Cycle End', 'Recipes',
    'Pre Rinse', 'Caustic', 'Acid', 'Final Rinse',
    'Supply Flow Actual (L/hr)', 'Return Flow Actual (L/hr)',
    'Alert Count', 'Status']
cycle_results[cycle_summary_columns].head(20)


,Line,Object,Reference Object,Object Key,Cycle Number,Cycle Start,Cycle End,Recipes,Pre Rinse,Caustic,Acid,Final Rinse,Supply Flow Actual (L/hr),Return Flow Actual (L/hr),Alert Count,Status
0,M01W01,C01C01-RMT Tank-1,NaN,C01C01,0,2026-07-02 19:38:06.703,2026-07-02 20:14:06.703,Caustic,5.000000,23.000000,NaN,5.000000,12784.3800,20220.0,0,NO REFERENCE
1,M01W01,C01C01-RMT Tank-1,NaN,C01C01,1,2026-07-04 00:43:06.703,2026-07-04 01:36:06.653,Caustic,5.000000,38.016667,NaN,6.983333,11404.6900,267.5,0,NO REFERENCE
2,M01W01,C01C01-RMT Tank-1,NaN,C01C01,2,2026-07-04 19:54:06.703,2026-07-04 21:14:06.653,Caustic Acid,7.983333,37.000000,12.000000,10.000000,12403.1300,3115.0,0,NO REFERENCE
3,M01W01,C01C01-RMT Tank-1,NaN,C01C01,3,2026-07-06 07:06:06.703,2026-07-06 08:14:06.703,Caustic,4.000000,51.983333,NaN,10.016667,862.5000,-45.0,0,NO REFERENCE
4,M01W01,C01C01-RMT Tank-1,NaN,C01C01,4,2026-07-07 06:19:06.703,2026-07-07 07:19:06.703,Caustic,5.000000,46.000000,NaN,7.000000,281.2500,-45.0,0,NO REFERENCE
5,M01W01,C01C01-RMT Tank-1,NaN,C01C01,5,2026-07-08 07:32:06.703,2026-07-08 08:10:06.653,Caustic,7.000000,23.000000,NaN,6.000000,12643.7500,11405.0,0,NO REFERENCE
6,M01W01,C01C01-RMT Tank-1,NaN,C01C01,6,2026-07-09 14:57:06.703,2026-07-09 15:40:06.703,Caustic,5.983333,29.016667,NaN,5.000000,12745.3100,18422.5,0,NO REFERENCE
7,M01W01,C01C01-RMT Tank-1,NaN,C01C01,7,2026-07-10 21:49:06.653,2026-07-10 22:52:06.703,"Caustic, Caustic Acid",5.016667,25.016667,11.000000,7.016667,12748.4400,15887.5,0,NO REFERENCE
8,M01W01,C01C01-RMT Tank-1,NaN,C01C01,8,2026-07-12 09:04:06.653,2026-07-12 09:48:06.703,Caustic,4.000000,32.016667,NaN,6.000000,11703.1300,-40.0,0,NO REFERENCE
9,M01W01,C01C01-RMT Tank-1,NaN,C01C01,9,2026-07-12 17:47:06.703,2026-07-12 17:55:06.653,"Caustic, Final rinse",0.000000,NaN,NaN,5.000000,8087.5000,-40.0,0,NO REFERENCE


In [14]:
alert_rows = []
for _, row in cycle_results.iterrows():
    base = row[cycle_keys + ['Cycle Start', 'Cycle End']].to_dict()
    for phase, (actual_col, threshold_col) in actual_to_threshold.items():
        if bool(row[f'{phase} Breach']):
            alert_rows.append({**base, 'Alert Type': f'{phase} duration',
                               'Actual Value': row[actual_col], 'Threshold': row[f'{phase} Threshold (min)'],
                               'Unit': 'minutes'})
    for signal, actual_col, threshold_col in [
        ('Supply flow', 'Supply Flow Actual (L/hr)', 'Supply Flow Threshold (L/hr)'),
        ('Return flow', 'Return Flow Actual (L/hr)', 'Return Flow Threshold (L/hr)')
    ]:
        if bool(row[f'{signal.title()} Breach']):
            alert_rows.append({**base, 'Alert Type': signal,
                               'Actual Value': row[actual_col], 'Threshold': row[threshold_col],
                               'Unit': 'L/hr'})

alerts = pd.DataFrame(alert_rows)
if alerts.empty:
    alerts = pd.DataFrame(columns=cycle_keys + ['Cycle Start', 'Cycle End', 'Alert Type', 'Actual Value', 'Threshold', 'Unit'])

print(f'{len(cycle_results):,} cycles identified')
print(f'{len(alerts):,} threshold breaches found')
alerts.head(50)


699 cycles identified
304 threshold breaches found


,Line,Object,Reference Object,Object Key,Cycle Number,Cycle Start,Cycle End,Alert Type,Actual Value,Threshold,Unit
0,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,1,2026-07-02 07:54:06.653,2026-07-02 08:53:06.653,Pre Rinse duration,15.983333,14.500000,minutes
1,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,5,2026-07-05 00:03:06.653,2026-07-05 03:17:06.703,Acid duration,115.783333,24.500000,minutes
2,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,13,2026-07-09 18:48:06.703,2026-07-09 20:37:06.703,Caustic duration,90.000000,33.500000,minutes
3,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,14,2026-07-10 09:01:06.653,2026-07-10 10:25:06.703,Pre Rinse duration,31.000000,14.500000,minutes
4,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,14,2026-07-10 09:01:06.653,2026-07-10 10:25:06.703,Caustic duration,42.000000,33.500000,minutes
5,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,14,2026-07-10 09:01:06.653,2026-07-10 10:25:06.703,Return flow,6637.500000,7425.000000,L/hr
6,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,17,2026-07-12 09:45:06.703,2026-07-12 10:44:06.653,Caustic duration,38.016667,33.500000,minutes
7,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,20,2026-07-13 20:53:06.703,2026-07-13 23:30:06.653,Caustic duration,88.983333,33.500000,minutes
8,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,20,2026-07-13 20:53:06.703,2026-07-13 23:30:06.653,Acid duration,31.983333,24.500000,minutes
9,M11W01,C01W01-Milk Past-1,M11W01_12_C01W01_MILK PAST-1,C01W01,21,2026-07-14 07:58:06.703,2026-07-14 10:08:06.703,Caustic duration,111.000000,33.500000,minutes


In [15]:
unmatched = (cycle_results.loc[~cycle_results['Reference Object'].isin(threshold_lookup.index),
                             ['Line', 'Object', 'Object Key']]
             .drop_duplicates()
             .sort_values(['Line', 'Object']))

cycle_counts = (cycle_results.groupby(['Line', 'Object'], as_index=False)
                .agg(Cycles=('Cycle Number', 'nunique'))
                .sort_values(['Line', 'Object']))
print('Cycles per object:')
cycle_counts.head(50)

print('Objects without a matching reference threshold:')
unmatched


Cycles per object:
Objects without a matching reference threshold:


,Line,Object,Object Key
0,M01W01,C01C01-RMT Tank-1,C01C01
21,M01W01,C01C02-RMT Tank-2,C01C02
44,M01W01,C01C03-RMT Tank-3,C01C03
71,M01W01,C01W03-RMT To Reception line,C01W03
92,M01W01,R01W01-Milk Reception Line,R01W01-MILK RECEPTION LINE
116,M01W02,M01W02_CC01 - Raw Milk Tanker,M01W02_CC01 - RAW MILK TANKER
421,M11W02,C01W02-Milk Past-2,C01W02


## How to use this notebook

Run the cells from top to bottom. The main outputs are `cycle_results`, which contains one row per object cycle, and `alerts`, which contains one row per breach.

The current flow comparison uses the median flow observed during a cycle. That avoids triggering an alert from one bad sensor reading. The duration and flow tolerances are working defaults, not confirmed plant rules. Confirm them with the CIP owner before treating the alerts as operational alarms.


In [ ]:
cycle_days = cycle_results.assign(**{'Alert Date': pd.to_datetime(cycle_results['Cycle Start']).dt.date})
daily_cycles = (cycle_days.groupby('Alert Date').size().rename('Cycles').to_frame())

alert_days = alerts.assign(**{'Alert Date': pd.to_datetime(alerts['Cycle Start']).dt.date})
daily_alerts = (alert_days.groupby('Alert Date')
                .agg(**{'Alert Count': ('Alert Type', 'size'),
                       'Cycles with Alerts': ('Cycle Number', 'nunique')}))

alerts_per_day = (daily_cycles.join(daily_alerts, how='left')
                  .fillna({'Alert Count': 0, 'Cycles with Alerts': 0})
                  .reset_index())
alerts_per_day['Alert Count'] = alerts_per_day['Alert Count'].astype(int)
alerts_per_day['Cycles with Alerts'] = alerts_per_day['Cycles with Alerts'].astype(int)
alerts_per_day.sort_values('Alert Date')
